# Notebook 2 — Optimisation numérique dans DICE

Ce notebook traite de la tarification du carbone et du contrôle optimal. Nous passons des scénarios — ce qui pourrait arriver — au problème du planificateur social — ce qui devrait arriver — puis calculons numériquement le coût social du carbone (CSC).

**Programme**
1. Charger le modèle et examiner les états et les contrôles.
2. Définir l’objectif du planificateur et l’approximation à horizon fini.
3. Résoudre le modèle avec une fenêtre glissante et lire le CSC.
4. Étudier la sensibilité aux dommages et exporter les résultats.


## 1) Préparation

Nous importons DICE et vérifions que Python trouve le module local.


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

p = Params()
print("DICE parameters loaded; periods:", p.nT)


## 2) Structure du modèle

- **États :** capital $K_t$, productivité $A_t$, population $L_t$, stocks de carbone $M_t^{AT},M_t^{UP},M_t^{LO}$ et températures $T_t^{AT},T_t^{LO}$.
- **Contrôles :** taux d’épargne $s_t$ et taux de réduction des émissions $\mu_t$ ; ce dernier détermine implicitement le prix du carbone.
- **Unités :** $C_t$ est en milliers de milliards de dollars et $L_t$ en millions d’habitants, donc $c_t=1000C_t/L_t$ est en milliers de dollars par habitant.


In [ ]:

# Paramètres instantanés et construire un chemin initial
p = Params()
sim = init_states(p)

# Contrôles de référence pour l'initialisation (pas optimaux): petite réduction, économie constante
sim[:, p.i_mu] = 0.03
sim[:, p.i_s]  = 0.20

# Mise à jour du chemin pour toutes les variables endogènes compte tenu des contrôles
timevec = range(1, p.nT)
sim = update_path(sim, timevec, p)

# Peek at available columns
print("Columns:", p.col)
print("Longueur de l’horizon (périodes) :", p.nT, " — années:", sim[-1, p.i_time])


## 3) Objectif du planificateur

Nous maximisons le bien-être social actualisé
$$
W=\sum_{i=0}^{I^\star}\beta_\Delta^iL_{t+i\Delta}u\!\left(1000\frac{C_{t+i\Delta}}{L_{t+i\Delta}}\right),\qquad \beta_\Delta=(1+\rho)^{-\Delta},
$$
avec une utilité CRRA $u(c)=(c^{1-\gamma}-1)/(1-\gamma)$. Le module fournit la fonction `obj_fun`.

L’horizon infini est remplacé par un horizon fini $I^\star$ dont la contribution résiduelle est inférieure à une tolérance.


In [ ]:

# Examiner l'objectif sur un chemin de contrôle constant (contrôle de la sanité)
# Ici, nous 'flattons' un contrôle (par exemple, l'épargne seulement) et évaluons le bien-être.
x_const = np.full(p.nT-1, 0.20)  # constant saving
W_val = obj_fun(x_const, sim, timevec, p, [p.i_s])
print("Objective value (negative welfare) on constant s=0.20 path:", W_val)


## 4) Fenêtre de planification à horizon fini $I^\star$

La fenêtre est choisie de façon que le poids d’actualisation du dernier terme soit inférieur à `p.toly`.


In [ ]:

# Calculer une fenêtre de planification analogue à celle de DICE.run_optimal_policy
disc, Tplanner = 1.0, 1
while disc > p.toly:
    Tplanner += 1
    disc *= (1.0/(1.0 + p.rho))**p.Delta

print("Fenêtre de planification retenue (périodes) :", Tplanner, " (each period = Δ années =", p.Delta, ")")


## 5) Premier problème : optimiser seulement l’épargne

Nous optimisons $s_t$ dans ses bornes et maintenons $\mu_t$ constant. Cette solution constitue la référence sans transition.


In [ ]:

print("Résolution : trajectoire optimale de l’épargne...")
bounds_s   = (p.s_lower, p.s_upper)
control_id = [p.i_s]
path_opt_s = run_optimal_policy(sim.copy(), timevec, p, bounds_s, control_id)

print("Done. Preview of s_t path (first 5):", path_opt_s[:5, p.i_s])


## 6) Optimiser l’épargne et la réduction des émissions

Nous optimisons maintenant $(s_t,\mu_t)$. Le prix implicite du carbone est stocké dans la colonne `Tax`.


In [ ]:

print("Résolution : trajectoire optimale jointe de (s_t, μ_t)...")
bounds_smu   = [(p.s_lower, p.s_upper), (0.0, 1.0)]
control_id   = [p.i_s, p.i_mu]
path_opt_smu = run_optimal_policy(sim.copy(), timevec, p, bounds_smu, control_id)

print("Done. Preview of μ_t path (first 5):", path_opt_smu[:5, p.i_mu])


## 7) Visualiser les contrôles et le prix du carbone

Tracez séparément l’épargne optimale $s_t$, le taux de réduction $\mu_t$ et le prix implicite du carbone `Tax`, en USD/tCO₂.


In [ ]:
# Taux d'épargne du lot
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_s], label="Optimal transition (s_t)")
plt.plot(path_opt_s[:,  p.i_time], path_opt_s[:,  p.i_s],  linestyle="--", label="No transition (s_t)")
plt.xlabel("Année"); plt.ylabel("Saving rate s_t"); plt.title("Optimal saving rate")
plt.legend(); plt.grid(True); plt.tight_layout()


In [ ]:
# Taux de réduction des parcelles
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_mu], label="Optimal transition (μ_t)")
plt.xlabel("Année"); plt.ylabel("Abatement rate μ_t"); plt.title("Optimal abatement rate")
plt.legend(); plt.grid(True); plt.tight_layout()


In [ ]:
# Taxe sur le carbone implicite (USD par tCO2)
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_Tax], label="Implied carbon tax (USD/tCO₂)")
plt.xlabel("Année"); plt.ylabel("Carbon tax (USD/tCO₂)"); plt.title("Optimal carbon price / SCC (model-implied)")
plt.legend(); plt.grid(True); plt.tight_layout()


## 8) Lire le coût social du carbone

- La colonne `Tax` contient le prix optimal du carbone, en USD/tCO₂.
- Le CSC est le prix d’ombre des émissions dans le problème du planificateur. Il peut aussi être estimé par une petite perturbation des émissions puis conversion de la perte marginale de bien-être en dollars.

Le squelette ci-dessous illustre cette seconde méthode ; il reste commenté pour réduire le temps d’exécution.


In [ ]:

# -- Optional: perturbation-based SCC at a chosen date (slow; left as a template) --
# t_idx = 10         # pick a period index
# eps = 1e-3 # ajouter 0,001 GtC (ajuster les unités au besoin) aux émissions à t idx
# 
# base = path_opt_smu.copy()
# pert = path_opt_smu.copy()
# 
# # Injecter une petite augmentation des émissions à t idx en réduisant la réduction d'un cran
# # (Ici comme une illustration; dans une expérience complète vous modifieriez E t directement ou ajouteriez au cycle du carbone)
# pert[t_idx, p.i_mu] = max(0.0, pert[t_idx, p.i_mu] - 1e-5)
# 
# base_W = obj_fun(np.hstack([base[1:, p.i_s], base[1:, p.i_mu]]), base, range(1, p.nT), p, [p.i_s, p.i_mu])
# pert_W = obj_fun(np.hstack([pert[1:, p.i_s], pert[1:, p.i_mu]]), pert, range(1, p.nT), p, [p.i_s, p.i_mu])
# dW = pert_W - base_W  # (negative welfare) difference
# print("ΔW de ε-perturbation:", dW)
# # Convertir en USD/tCO2 en utilisant l'utilitaire marginal à t idx si nécessaire.


## 9) Sensibilité à des dommages plus élevés

Modifiez la fonction de dommages avec `p.user_damage_fn`, puis relancez l’optimisation pour observer la réaction de $\mu_t$ et du prix du carbone.


In [ ]:

def higher_damage(T, p_local: Params):
    # Slightly steeper quadratic damages as an illustration
    return 0.0030 * (np.asarray(T) ** 2)

p_sens = Params()
sim_sens = init_states(p_sens)
sim_sens[:, p_sens.i_mu] = 0.03
sim_sens[:, p_sens.i_s]  = 0.20
sim_sens = update_path(sim_sens, range(1, p_sens.nT), p_sens)

p_sens.user_damage_fn = higher_damage
path_opt_smu_highDam = run_optimal_policy(sim_sens.copy(), range(1, p_sens.nT), p_sens, [(p_sens.s_lower,p_sens.s_upper),(0.0,1.0)], [p_sens.i_s, p_sens.i_mu])

# Comparaison par parcelle pour μ t
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_mu], linestyle="--", label="Baseline damages")
plt.plot(path_opt_smu_highDam[:, p.i_time], path_opt_smu_highDam[:, p.i_mu], label="Higher damages")
plt.xlabel("Année"); plt.ylabel("Abatement rate μ_t"); plt.title("Damage sensitivity: optimal abatement")
plt.legend(); plt.grid(True); plt.tight_layout()

# Comparaison par parcelle pour la taxe implicite sur le carbone
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_Tax], linestyle="--", label="Baseline damages")
plt.plot(path_opt_smu_highDam[:, p.i_time], path_opt_smu_highDam[:, p.i_Tax], label="Higher damages")
plt.xlabel("Année"); plt.ylabel("Carbon tax (USD/tCO₂)"); plt.title("Damage sensitivity: optimal carbon price")
plt.legend(); plt.grid(True); plt.tight_layout()


## 10) Exporter les résultats

Convertissez les tableaux en `DataFrame` pour poursuivre l’analyse ou produire d’autres graphiques.


In [ ]:

df_baseline = pd.DataFrame({
    "year": path_opt_smu[:, p.i_time],
    "s":    path_opt_smu[:, p.i_s],
    "mu":   path_opt_smu[:, p.i_mu],
    "tax":  path_opt_smu[:, p.i_Tax],
    "E":    path_opt_smu[:, p.i_E],
    "T_AT": path_opt_smu[:, p.i_T_AT],
    "C":    path_opt_smu[:, p.i_C],
})
df_baseline.head()


### Conclusion

- Le prix optimal `Tax` constitue la recommandation normative du modèle.
- La fenêtre finie met en œuvre la troncation temporelle.
- La résolution par fenêtre glissante produit les trajectoires complètes des états et des contrôles.
- Les hypothèses de dommages modifient directement $\mu_t$ et le prix implicite du carbone.
